# NB-09 — Multimodal RAG

**Goal:** Ground QA answers in a retrieved knowledge base using CLIP + FAISS.

Flow: **index** documents → **retrieve** by image/text → **augment** prompt → **generate** answer.

---

In [ ]:
import sys
import json
import time
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import torch
from dotenv import load_dotenv
from omegaconf import OmegaConf
from PIL import Image

load_dotenv(Path("..") / ".env")

from src.encoders.clip_encoder import CLIPVisionEncoder
from src.pipeline.multimodal_qa import MultimodalQAPipeline
from src.pipeline.rag_retriever import MultimodalRetriever, format_retrieval_context

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = Path("..") / "config" / "model_config.yaml"
KB_DIR = Path("..") / "data" / "rag_kb"
INDEX_DIR = Path("..") / "data" / "rag_index"
print(f"Device: {DEVICE}")

## 1. Build a knowledge base

Each entry: `{"image": path, "text": caption, "metadata": {...}}`.

Set `DOWNLOAD_COCO=False` for a quick local demo KB.

In [ ]:
DOWNLOAD_COCO = False
KB_DIR.mkdir(parents=True, exist_ok=True)
(KB_DIR / "images").mkdir(exist_ok=True)

if DOWNLOAD_COCO:
    from datasets import load_dataset

    ds = load_dataset("Multimodal-Fatima/COCO_captions", split="train[:40]")
    docs = []
    for i, ex in enumerate(ds):
        img_path = KB_DIR / "images" / f"{i:04d}.jpg"
        ex["image"].save(img_path)
        docs.append({
            "image": img_path,
            "text": ex["caption"],
            "metadata": {"id": i, "source": "coco"},
        })
else:
    entries = [
        ((255, 80, 80), "A red sports car on a city street"),
        ((80, 180, 80), "A green forest with tall trees"),
        ((80, 120, 220), "A blue ocean with waves"),
        ((220, 180, 60), "A golden retriever dog in a park"),
        ((180, 100, 220), "A purple sunset over mountains"),
    ]
    docs = []
    for i, (color, caption) in enumerate(entries * 8):
        img_path = KB_DIR / "images" / f"{i:03d}.jpg"
        if not img_path.exists():
            Image.new("RGB", (224, 224), color).save(img_path)
        docs.append({
            "image": img_path,
            "text": caption,
            "metadata": {"id": i, "topic": caption.split()[1]},
        })

print(f"Knowledge base size: {len(docs)}")

In [ ]:
encoder = CLIPVisionEncoder(device=DEVICE)
retriever = MultimodalRetriever(
    encoder,
    image_weight=0.5,
    text_weight=0.5,
)

t0 = time.perf_counter()
retriever.index_documents(docs)
print(f"Indexed in {time.perf_counter() - t0:.2f}s | docs={retriever.num_documents}")

retriever.save_index(INDEX_DIR)
print(f"Saved index to {INDEX_DIR}")

## 2. Embedding space visualization (UMAP)

In [ ]:
try:
    import umap

    embeddings = []
    labels = []
    for doc in docs[:30]:
        img = Image.open(doc["image"]).convert("RGB")
        emb = encoder._projected_image_features([img]).cpu().numpy()[0]
        embeddings.append(emb)
        labels.append(doc["text"][:20])

    reducer = umap.UMAP(n_components=2, random_state=42)
    coords = reducer.fit_transform(np.array(embeddings))

    plt.figure(figsize=(10, 6))
    plt.scatter(coords[:, 0], coords[:, 1], alpha=0.7)
    for i, label in enumerate(labels):
        plt.annotate(label, (coords[i, 0], coords[i, 1]), fontsize=7)
    plt.title("CLIP image embeddings (UMAP)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install umap-learn for embedding visualization.")

## 3. Retrieval demos — image and text queries

In [ ]:
def show_retrieval(results, title: str) -> None:
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, hit in zip(axes, results):
        if "_image" in hit:
            ax.imshow(hit["_image"])
        ax.set_title(f"{hit['score']:.3f}\n{hit['text'][:40]}", fontsize=8)
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

query_img = Image.open(docs[3]["image"]).convert("RGB")
img_hits = retriever.retrieve(query_image=query_img, top_k=3)
show_retrieval(img_hits, "Image query → top 3")

txt_hits = retriever.retrieve(query_text="ocean waves beach", top_k=3)
show_retrieval(txt_hits, "Text query → top 3")

## 4. RAG-augmented QA — with vs without retrieval

In [ ]:
RUN_LLM = False  # set True to load Qwen2-VL and compare answers

question = "Describe what kind of scene might match a dog in a park."
context = format_retrieval_context(
    retriever.retrieve(query_text="golden retriever dog park", top_k=3)
)
print("Retrieved context:\n", context[:600], "...\n")

if RUN_LLM:
    pipeline = MultimodalQAPipeline(
        CONFIG,
        device=DEVICE,
        encoder_on_cpu=(DEVICE == "mps"),
        retriever=retriever,
        rag_top_k=3,
    )
    query_img = Image.open(docs[3]["image"]).convert("RGB")

    t0 = time.perf_counter()
    no_rag = pipeline.answer(question, image=query_img, use_rag=False, max_new_tokens=128)
    t1 = time.perf_counter()
    with_rag = pipeline.answer(question, image=query_img, use_rag=True, max_new_tokens=128)
    t2 = time.perf_counter()

    print("Without RAG:", no_rag)
    print(f"Latency: {t1 - t0:.2f}s\n")
    print("With RAG:", with_rag)
    print(f"Latency: {t2 - t1:.2f}s")
else:
    print("Set RUN_LLM=True to compare base vs RAG answers with Qwen2-VL.")

## 5. Analysis — when does RAG help?

| Scenario | RAG helps | RAG hurts |
|----------|-----------|----------|
| Domain KB with factual captions | Grounding rare facts | — |
| Query matches indexed content | Higher precision | — |
| Noisy / wrong retrieval | — | Hallucination from bad context |
| Creative / open-ended tasks | — | Over-constrains the model |

Measure retrieval latency separately from generation — retrieval is typically &lt;100ms on CPU for small indexes.

In [ ]:
latencies = []
for _ in range(10):
    t0 = time.perf_counter()
    retriever.retrieve(query_text="forest trees green", top_k=3)
    latencies.append(time.perf_counter() - t0)

print(f"Retrieval latency: mean={np.mean(latencies)*1000:.1f}ms | p95={np.percentile(latencies, 95)*1000:.1f}ms")

---

**Phase 6 complete.** Next: Phase 7 — FastAPI + OpenWebUI (`src/serving/api_server.py`, `NB-10-deployment.ipynb`).